# 确认训练配置

启动训练前，先用 DRY_RUN 展开 launcher 中的环境变量和默认值，查看最终交给 Verl 的完整命令。这样可以一次核对后端、并行、注意力和 RL 参数；此时不会启动 Hydra，也不会占用 NPU 运行训练。


## 需要确认的配置

| 检查范围 | 预期内容 | 说明 |
| --- | --- | --- |
| 后端选择 | `model_engine=torchtitan` | Actor/Ref 使用 TorchTitan Engine |
| FSDP2 拓扑 | DP shard 为 2、CP 为 1、replicate 为 1 | 两卡分片训练状态 |
| 内存策略 | Actor 参数/优化器 offload 与 reshard、Ref 参数 offload | 对齐 Actor/Ref 的不同职责 |
| NPU 路径 | TND varlen、最大长度 5120 | 对齐 Qwen3 训练路径 |
| 算法 | GRPO | 保持初阶语义 |
| rollout | vLLM async、TP=2、六轮 Wordle AgentLoop | 保持初阶语义 |
| 数据与奖励 | 07.02 已准备的 parquet 与 `wordle_reward.py` | 保持 Wordle 任务语义 |


## 运行下面的 Cell

Cell 会以 DRY_RUN 模式调用 launcher，并打印 shell 展开后的启动命令。检查输出时，既要看到 TorchTitan-NPU、DP shard 2、CP 1 和 TND varlen，也要确认 GRPO、vLLM、Wordle AgentLoop 和奖励函数仍在命令中。

如果 Cell 在打印命令前中止，请回到 07.02 检查独立环境和训练资产，并确认当前训练代码包含 TorchTitan-NPU launcher。


In [ ]:
%%bash
set -euo pipefail

COURSE_ROOT=$(git rev-parse --show-toplevel)
TRAIN_DIR="$(dirname "${COURSE_ROOT}")/cann-recipes-train/llm_rl/qwen3_wordle"
cd "${TRAIN_DIR}"
DRY_RUN=1 \
    bash torchtitan_backend/run_qwen3_1.7b_wordle_torchtitan_npu.sh


看到完整命令后，环境预检和参数拼接就完成了。真正的配置解析、模型初始化和 Actor 更新要等训练启动后才会发生。下一节继续使用同一个 launcher，依次经过 rollout、奖励、GRPO、前反向、优化器更新和权重同步。


## 课后练习

### 判断题

1. （判断题）DRY_RUN 会输出最终训练命令，但不会进入 rollout 和反向传播。

2. （判断题）确认 `model_engine=torchtitan` 后，可以忽略 rollout 和奖励函数配置是否保持不变。

### 单选题

3. （单选题）DRY_RUN 输出中哪项配置直接选择了变长注意力路径？

   A. `attn_type=varlen`

   B. `context_parallel_size=1`

   C. `max_seq_len=5120`

   D. `rollout.name=vllm`

### 多选题

4. （多选题）哪些配置用于确认原有 RL 语义保持不变？

   A. `adv_estimator=grpo`

   B. `rollout.name=vllm`

   C. `default_agent_loop=wordle_agent`

   D. `custom_reward_function.path`

5. （多选题）哪些配置用于确认启动命令包含 FSDP2 + TND 训练路径？

   A. `data_parallel_shard_size=2`

   B. `context_parallel_size=1`

   C. `param_offload=True`

   D. `attn_type=varlen`


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/07_torchtitan_wordle_training/answer/07.03_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
